# Profile the movie data

This notebook is a repeatable **data-quality and exploratory analysis (EDA) recipe** for the local TMDB movie cache. It reads cached JSONL records, uses the project's normalizer, and never makes API requests or changes the source data.

## Profiling recipe

1. Load and normalize the cache so the profiling uses the same table definitions as the project.
2. Record each table's row count, columns, types, missing values, and declared key quality.
3. Search for semantic duplicate candidates by normalized title and release year, then review matches.
4. Review distributions and unusual values in movie measures and release dates.
5. Check how films connect to genres, companies, countries, languages, cast, and crew.
6. Investigate signals against the source records and intended business rules before deciding whether anything is actually invalid.

Run top to bottom. A value flagged as unusual is a prompt for investigation, not an instruction to delete or correct it.


In [1]:
from IPython.display import display
import pandas as pd

from streamvault.cache import load_cached_records
from streamvault.config import CONFIG
from streamvault.normalize import normalize_movie_records

records = list(load_cached_records(CONFIG.cache_file).values())
tables = normalize_movie_records(records)
movies = tables.movies

frames = {
    name: getattr(tables, name)
    for name in (
        "movies",
        "genres",
        "movie_genres",
        "companies",
        "movie_companies",
        "countries",
        "movie_origin_countries",
        "movie_production_countries",
        "languages",
        "movie_spoken_languages",
        "collections",
        "people",
        "cast_credits",
        "crew_credits",
    )
}

print(f"Cache: {CONFIG.cache_file}")
print(f"Cached movie records: {len(records):,}")
if not records:
    print("No cached movies yet. Run `uv run streamvault` and rerun this notebook.")
### I should run my checks again.


Cache: /home/faazeleishola/training/streamvault/data/raw/movies.jsonl
Cached movie records: 112


## 1. Table inventory and schema

Start with the grain and shape of each table. A link/bridge table has one row per relationship, so it can have more rows than the movie table.


In [2]:
inventory = pd.DataFrame(
    [
        {"table": name, "rows": len(frame), "columns": len(frame.columns)}
        for name, frame in frames.items()
    ]
)
display(inventory)

schema_rows = []
for table_name, frame in frames.items():
    for column in frame.columns:
        values = frame[column]
        schema_rows.append(
            {
                "table": table_name,
                "column": column,
                "dtype": str(values.dtype),
                "missing": int(values.isna().sum()),
                "missing_percent": (
                    round(values.isna().mean() * 100, 2) if len(frame) else pd.NA
                ),
                "distinct_non_missing": int(values.nunique(dropna=True)),
            }
        )

schema_profile = pd.DataFrame(schema_rows)

import ipywidgets as widgets

page_size = 15
page_count = (len(schema_profile) + page_size - 1) // page_size
if page_count == 0:
    print("No schema rows to display.")
else:
    page_state = {"index": 0}
    previous_button = widgets.Button(description="Previous", icon="chevron-left")
    next_button = widgets.Button(description="Next", icon="chevron-right")
    page_label = widgets.Label()
    page_output = widgets.Output()

    def render_schema_page():
        page_index = page_state["index"]
        start = page_index * page_size
        end = min(start + page_size, len(schema_profile))
        with page_output:
            page_output.clear_output(wait=True)
            display(schema_profile.iloc[start:end])
        page_label.value = (
            f"Page {page_index + 1} of {page_count} "
            f"(rows {start + 1}-{end} of {len(schema_profile)})."
        )
        previous_button.disabled = page_index == 0
        next_button.disabled = page_index == page_count - 1

    def show_previous_page(_):
        page_state["index"] -= 1
        render_schema_page()

    def show_next_page(_):
        page_state["index"] += 1
        render_schema_page()

    previous_button.on_click(show_previous_page)
    next_button.on_click(show_next_page)
    display(widgets.HBox([previous_button, page_label, next_button]))
    display(page_output)
    render_schema_page()


,table,rows,columns
0,movies,112,22
1,genres,16,2
2,movie_genres,320,2
3,companies,136,4
4,movie_companies,307,2
5,countries,12,2
6,movie_origin_countries,117,2
7,movie_production_countries,154,2
8,languages,27,3
9,movie_spoken_languages,219,2


Output()

- `Adult` All movies are tagged as False on the adult field. Provides no predictive value
- `budget` 60 distinct budget with no missing value is surprising. Dataset may have been filtered towards well known movies with complete metadata.
- 2 home page entrys are duplicates
- `Original Language` is mostly english and one japanese and spanish.
- `runtime` is rounded given a fewer distinct values.
- `softscore` is similar to `adult` field.
- `status`: all films are `released` so it has no predictive power.
- `video` : no predictive power. all of them are movies
- `vote_avg`: need investigate, why some of them are the same for a float.
- `language.language_name` and `language_english_name` are redundant.

- All investigations have been carried out: runtime having fewer distinct value, vote_avg having fewer distinct values.
- no preditict power: `adult`, `original_language`, `softcore`, `status`, `video`, `language.language_name`

In [3]:
frames.get("movies")["adult"].value_counts()

adult
False    112
Name: count, dtype: int64

In [4]:
frames.get("movies")["budget"].describe()

count    1.120000e+02
mean     1.084460e+08
std      8.278818e+07
min      1.200000e+06
25%      3.808750e+07
50%      9.100000e+07
75%      1.500000e+08
max      3.790000e+08
Name: budget, dtype: float64

In [5]:
frames.get("movies")["original_language"].value_counts()

original_language
en    110
ja      1
it      1
Name: count, dtype: int64

In [6]:
frames.get("movies")["status"].value_counts()

status
Released    112
Name: count, dtype: int64

In [7]:
frames.get("movies")["collection_id"].value_counts()

collection_id
1241      8
10        6
295       5
87359     5
8091      4
1570      4
10194     4
86311     4
2150      4
84        4
528       4
263       3
264       3
230       3
328       3
131292    3
119       3
121938    3
2344      3
304       3
8354      3
31562     3
2883      2
94032     1
468222    1
9743      1
134897    1
422837    1
2366      1
137696    1
8945      1
2980      1
Name: count, dtype: Int64

In [8]:
languages = frames.get("languages")
missing = languages[languages["language_name"].isna()]
languages

,language_code,language_name,language_english_name
0,en,English,English
1,zh,普通话,Mandarin
2,it,Italiano,Italian
3,de,Deutsch,German
4,la,Latin,Latin
5,es,Español,Spanish
6,fr,Français,French
7,tr,Türkçe,Turkish
8,el,ελληνικά,Greek
9,ru,Pусский,Russian


In [9]:
people = frames.get("people")
missing = people[people["known_for_department"].isna()]
missing

,person_id,adult,gender,known_for_department,name,original_name,profile_path
17473,8168,False,2,NaN,Brian Magerkurth,Brian Magerkurth,NaN


In [10]:
people["known_for_department"].value_counts()

known_for_department
Acting               6198
Crew                 3215
Visual Effects       2457
Production           1935
Art                  1766
Sound                1536
Costume & Make-Up    1287
Camera               1118
Directing             827
Lighting              687
Editing               527
Writing               287
Creator                 3
Name: count, dtype: int64

In [11]:
movies = frames.get("movies")

vote_avg_counts = movies["vote_average"].value_counts()
dup_avgs = vote_avg_counts[vote_avg_counts > 1]

for vote_avg in dup_avgs.index:
    frame_x = movies.loc[
        movies["vote_average"] == vote_avg,
        ["title", "vote_average", "vote_count"]
    ]
    
    print(f"\nvote_average = {vote_avg} ({dup_avgs[vote_avg]} movies)")
    display(frame_x)


vote_average = 8.451 (2 movies)


,title,vote_average,vote_count
14,GoodFellas,8.451,15041
18,The Lord of the Rings: The Fellowship of the Ring,8.451,28890



vote_average = 7.9 (2 movies)


,title,vote_average,vote_count
20,Return of the Jedi,7.9,17566
91,Indiana Jones and the Last Crusade,7.9,11543



vote_average = 8.247 (2 movies)


,title,vote_average,vote_count
27,Avengers: Endgame,8.247,29384
51,Avengers: Infinity War,8.247,33418


... (2 outputs left out)


vote_average = 7.818 (2 movies)


,title,vote_average,vote_count
47,Die Hard,7.818,12876
86,Toy Story 3,7.818,16456



vote_average = 8.2 (2 movies)


,title,vote_average,vote_count
64,Alien,8.2,17148
75,Memento,8.2,17054


In [12]:
runtime_counts = movies["runtime"].value_counts()
dup_runtimes = runtime_counts[runtime_counts > 1]

for runtime in dup_runtimes.index:
    frame_x = movies.loc[
        movies["runtime"] == runtime,
        ["title", "runtime", "vote_average", "vote_count", "tagline", "release_date"]
    ]

    print(f"\nruntime = {runtime} minutes ({dup_runtimes[runtime]} movies)")
    display(frame_x)


runtime = 92 minutes (4 movies)


,title,runtime,vote_average,vote_count,tagline,release_date
5,Jurassic Park III,92,6.179,8195,"This time, it's not just a walk in the park!",2001-07-18
26,Toy Story 2,92,7.623,15594,The toys are back!,1999-10-30
76,Shrek 2,92,7.333,13982,What happens after happily ever after?,2004-05-19
88,"Monsters, Inc.",92,7.860,20528,It's nothing personal. It's just their job.,2001-11-01



runtime = 124 minutes (4 movies)


,title,runtime,vote_average,vote_count,tagline,release_date
8,Iron Man 2,124,6.856,23248,"It's not the armor that makes the hero, but th...",2010-04-28
13,Die Hard 2,124,6.971,6718,Die harder.,1990-07-03
49,The Empire Strikes Back,124,8.392,19137,The Star Wars saga continues.,1980-05-20
85,Jaws,124,7.687,12039,The terrifying motion picture from the terrify...,1975-06-20



runtime = 136 minutes (4 movies)


,title,runtime,vote_average,vote_count,tagline,release_date
31,Star Wars: Episode I - The Phantom Menace,136,6.582,16435,Every saga has a beginning.,1999-05-19
35,The Matrix,136,8.263,29272,Believe the unbelievable.,1999-03-31
73,Pirates of the Caribbean: On Stranger Tides,136,6.576,15449,Live forever or die trying.,2011-05-15
99,Kill Bill: Vol. 2,136,7.872,15687,The bride is back for the final cut.,2004-04-16


... (58 outputs left out)


runtime = 140 minutes (2 movies)


,title,runtime,vote_average,vote_count,tagline,release_date
104,Star Wars: Episode III - Revenge of the Sith,140,7.479,15516,Every story has a hero. Every hero has a desti...,2005-05-17
105,Batman Begins,140,7.733,23544,Evil fears the knight.,2005-06-10



runtime = 109 minutes (2 movies)


,title,runtime,vote_average,vote_count,tagline,release_date
110,Terminator 3: Rise of the Machines,109,6.216,7641,The machines will rise.,2003-07-02
111,Alien Resurrection,109,6.170,5877,It's already too late.,1997-11-12


## 1b. Check table keys

These are descriptive checks against each table's declared grain. They show duplicate and null key rows but do not alter or discard records. The normalizer already rejects several malformed or duplicate inputs; these checks keep the output visible at the analysis stage too.


In [13]:
key_columns = {
    "movies": ["movie_id"],
    "genres": ["genre_id"],
    "movie_genres": ["movie_id", "genre_id"],
    "companies": ["company_id"],
    "movie_companies": ["movie_id", "company_id"],
    "countries": ["country_code"],
    "movie_origin_countries": ["movie_id", "country_code"],
    "movie_production_countries": ["movie_id", "country_code"],
    "languages": ["language_code"],
    "movie_spoken_languages": ["movie_id", "language_code"],
    "collections": ["collection_id"],
    "people": ["person_id"],
    "cast_credits": ["credit_id"],
    "crew_credits": ["credit_id"],
}

key_quality_rows = []
for table_name, columns in key_columns.items():
    frame = frames[table_name]
    key_quality_rows.append(
        {
            "table": table_name,
            "key": " + ".join(columns),
            "rows_with_null_key": int(frame[columns].isna().any(axis=1).sum()),
            "duplicate_key_rows": int(frame.duplicated(subset=columns).sum()),
        }
    )

key_quality = pd.DataFrame(key_quality_rows)
display(key_quality)


,table,key,rows_with_null_key,duplicate_key_rows
0,movies,movie_id,0,0
1,genres,genre_id,0,0
2,movie_genres,movie_id + genre_id,0,0
3,companies,company_id,0,0
4,movie_companies,movie_id + company_id,0,0
5,countries,country_code,0,0
6,movie_origin_countries,movie_id + country_code,0,0
7,movie_production_countries,movie_id + country_code,0,0
8,languages,language_code,0,0
9,movie_spoken_languages,movie_id + language_code,0,0


## 1c. Semantic duplicate candidates

Unique `movie_id` values do not rule out the same real film appearing under different IDs. This search groups by a title normalized for case and whitespace plus the release year. It excludes rows without a usable title or release year, prints all candidate records with useful comparison fields, and never drops anything. A matching group is only a lead: same-title films and separate works released in the same year must be reviewed rather than assumed duplicates.


In [14]:
semantic_review = movies.copy()
if "title" in semantic_review.columns:
    normalized_title = (
        semantic_review["title"]
        .astype("string")
        .str.strip()
        .str.casefold()
        .str.replace(r"\s+", " ", regex=True)
    )
else:
    semantic_review["title"] = pd.Series(pd.NA, index=semantic_review.index, dtype="string")
    normalized_title = pd.Series(pd.NA, index=semantic_review.index, dtype="string")

if "release_date" in semantic_review.columns:
    release_year = semantic_review["release_date"].dt.year
else:
    release_year = pd.Series(pd.NA, index=semantic_review.index, dtype="Int64")

semantic_review["normalized_title"] = normalized_title
semantic_review["release_year"] = release_year
has_semantic_key = (
    semantic_review["normalized_title"].notna()
    & semantic_review["normalized_title"].ne("")
    & semantic_review["release_year"].notna()
).fillna(False)
eligible_rows = semantic_review.loc[has_semantic_key]
candidate_mask = eligible_rows.duplicated(
    subset=["normalized_title", "release_year"], keep=False
)
semantic_suspects = eligible_rows.loc[candidate_mask].copy()

semantic_candidate_groups = (
    semantic_suspects.groupby(
        ["normalized_title", "release_year"], as_index=False
    )
    .agg(
        movie_count=("movie_id", "size"),
        movie_ids=("movie_id", list),
        titles=("title", list),
    )
    .sort_values(["movie_count", "normalized_title"], ascending=[False, True])
)

review_columns = [
    column
    for column in (
        "movie_id",
        "title",
        "original_title",
        "release_date",
        "imdb_id",
        "collection_id",
    )
    if column in semantic_suspects.columns
]
print(
    f"{len(semantic_suspects)} rows in {len(semantic_candidate_groups)} "
    f"candidate groups; {int(has_semantic_key.sum())} rows eligible; "
    f"{int((~has_semantic_key).sum())} rows missing title or release year."
)
display(semantic_candidate_groups)
display(
    semantic_suspects[
        review_columns + ["normalized_title", "release_year"]
    ].sort_values(["normalized_title", "release_year", "movie_id"])
)


0 rows in 0 candidate groups; 112 rows eligible; 0 rows missing title or release year.


,normalized_title,release_year,movie_count,movie_ids,titles


,movie_id,title,original_title,release_date,imdb_id,collection_id,normalized_title,release_year


## 2. Data Types: Validation, Downcasting and Encoding

In [15]:
from pandas.api.types import is_integer_dtype

if movies.empty:
    print("No movie rows available to benchmark.")
else:
    original_movies = movies.copy(deep=True)
    benchmark_movies = movies.copy(deep=True)
    memory_stages = []

    def record_memory(stage, frame):
        memory_bytes = int(frame.memory_usage(index=True, deep=True).sum())
        previous_bytes = memory_stages[-1]["memory_bytes"] if memory_stages else None
        baseline_bytes = memory_stages[0]["memory_bytes"] if memory_stages else memory_bytes
        memory_stages.append(
            {
                "stage": stage,
                "memory_bytes": memory_bytes,
                "memory_kib": round(memory_bytes / 1024, 2),
                "saved_bytes_vs_previous": (
                    pd.NA if previous_bytes is None else previous_bytes - memory_bytes
                ),
                "saved_percent_vs_baseline": round(
                    (baseline_bytes - memory_bytes) / baseline_bytes * 100, 2
                ) if baseline_bytes else 0.0,
            }
        )

    record_memory("Baseline: normalized movies", benchmark_movies)

    dtype_change_rows = []
    integer_columns = [
        column
        for column in benchmark_movies.columns
        if is_integer_dtype(benchmark_movies[column].dtype)
    ]
    for column in integer_columns:
        before = benchmark_movies[column]
        observed = before.dropna()
        if observed.empty:
            continue

        downcast_kind = "unsigned" if bool(observed.ge(0).all()) else "integer"
        after = pd.to_numeric(before, downcast=downcast_kind)
        if not before.astype("string").equals(after.astype("string")):
            raise AssertionError(f"Downcasting changed values in {column!r}.")

        benchmark_movies[column] = after
        dtype_change_rows.append(
            {
                "column": column,
                "before_dtype": str(before.dtype),
                "after_dtype": str(after.dtype),
                "minimum": observed.min(),
                "maximum": observed.max(),
                "missing_values": int(before.isna().sum()),
                "downcast_kind": downcast_kind,
            }
        )
        record_memory(f"After integer downcast: {column}", benchmark_movies)

    category_columns = [
        column
        for column in ("original_language", "status")
        if column in benchmark_movies.columns
    ]
    for column in category_columns:
        before = benchmark_movies[column]
        after = before.astype("category")
        if not before.astype("string").equals(after.astype("string")):
            raise AssertionError(f"Categorical encoding changed values in {column!r}.")

        benchmark_movies[column] = after
        dtype_change_rows.append(
            {
                "column": column,
                "before_dtype": str(before.dtype),
                "after_dtype": str(after.dtype),
                "minimum": pd.NA,
                "maximum": pd.NA,
                "missing_values": int(before.isna().sum()),
                "downcast_kind": "category",
            }
        )
        record_memory(f"After categorical encoding: {column}", benchmark_movies)

    if not movies.equals(original_movies):
        raise AssertionError("The experiment unexpectedly changed the normalized movies DataFrame.")

    print("Memory includes the DataFrame index and deep memory for values.")
    print("Positive saved-byte values indicate a reduction from the preceding stage.")
    display(pd.DataFrame(memory_stages))
    display(pd.DataFrame(dtype_change_rows))

    raw_release_dates_by_id = {
        int(record["id"]): record.get("release_date")
        for record in records
    }
    source_release_dates = movies["movie_id"].map(raw_release_dates_by_id).astype("string")
    parsed_release_dates = pd.to_datetime(
        source_release_dates,
        format="%Y-%m-%d",
        errors="coerce",
    )
    invalid_date_mask = source_release_dates.notna() & parsed_release_dates.isna()
    if invalid_date_mask.any():
        bad_movie_ids = movies.loc[invalid_date_mask, "movie_id"].tolist()
        raise ValueError(f"Invalid source release dates for movie IDs: {bad_movie_ids}")

    dates_match_normalized = (
        parsed_release_dates.eq(movies["release_date"])
        | (parsed_release_dates.isna() & movies["release_date"].isna())
    )
    if not bool(dates_match_normalized.fillna(False).all()):
        raise AssertionError("Notebook-parsed release dates differ from normalized values.")

    date_representation = pd.DataFrame(
        [
            {
                "representation": "Source date strings",
                "dtype": str(source_release_dates.dtype),
                "memory_bytes": int(source_release_dates.memory_usage(index=False, deep=True)),
            },
            {
                "representation": "Parsed datetime",
                "dtype": str(parsed_release_dates.dtype),
                "memory_bytes": int(parsed_release_dates.memory_usage(index=False, deep=True)),
            },
        ]
    )
    date_representation["memory_kib"] = (
        date_representation["memory_bytes"] / 1024
    ).round(2)
    print("Release-date representation comparison (column only):")
    display(date_representation)
    print(
        f"Parsed {int(source_release_dates.notna().sum()):,} non-missing source dates; "
        "all parsed values match the normalized dates."
    )
    print(
        "Runtime is not benchmarked: use a larger, representative cache and repeated "
        "timings before drawing runtime conclusions."
    )


Memory includes the DataFrame index and deep memory for values.
Positive saved-byte values indicate a reduction from the preceding stage.


,stage,memory_bytes,memory_kib,saved_bytes_vs_previous,saved_percent_vs_baseline
0,Baseline: normalized movies,140626,137.33,<NA>,0.00
1,After integer downcast: movie_id,140178,136.89,448,0.32
2,After integer downcast: budget,139730,136.46,448,0.64
3,After integer downcast: revenue,139282,136.02,448,0.96
4,After integer downcast: runtime,138498,135.25,784,1.51
5,After integer downcast: vote_count,137826,134.60,672,1.99
6,After integer downcast: collection_id,137378,134.16,448,2.31
7,After categorical encoding: original_language,131931,128.84,5447,6.18
8,After categorical encoding: status,125716,122.77,6215,10.60


,column,before_dtype,after_dtype,minimum,maximum,missing_values,downcast_kind
0,movie_id,int64,uint32,11,301528,0,unsigned
1,budget,int64,uint32,1200000,379000000,0,unsigned
2,revenue,int64,uint32,7390108,2942649778,0,unsigned
3,runtime,int64,uint8,81,202,0,unsigned
4,vote_count,int64,uint16,4136,41925,0,unsigned
5,collection_id,Int64,UInt32,10,468222,16,unsigned
6,original_language,str,category,<NA>,<NA>,0,category
7,status,str,category,<NA>,<NA>,0,category


Release-date representation comparison (column only):


,representation,dtype,memory_bytes,memory_kib
0,Source date strings,string,6608,6.45
1,Parsed datetime,datetime64[us],896,0.88


Parsed 112 non-missing source dates; all parsed values match the normalized dates.
Runtime is not benchmarked: use a larger, representative cache and repeated timings before drawing runtime conclusions.


## 3. Numeric movie fields and unusual values

Use quantiles and interquartile-range (IQR) bounds to locate values worth checking. These bounds are exploratory, not universal validity rules; budget and revenue may be zero or absent, for example, and an extreme value may be legitimate.


In [16]:
identifier_columns = {"movie_id", "collection_id"}
numeric_movie_columns = [
    column
    for column in movies.select_dtypes(include="number").columns
    if column not in identifier_columns
]

if numeric_movie_columns:
    numeric_summary = movies[numeric_movie_columns].describe(
        percentiles=[0.01, 0.25, 0.50, 0.75, 0.90, 0.99]
    ).T
    display(numeric_summary)
else:
    print("No numeric movie measures are available.")

outlier_rows = []
for column in numeric_movie_columns:
    values = pd.to_numeric(movies[column], errors="coerce")
    valid_values = values.dropna()
    if valid_values.empty:
        continue

    q1 = valid_values.quantile(0.25)
    q3 = valid_values.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    unusual = values.lt(lower_bound) | values.gt(upper_bound)
    examples = movies.loc[unusual, ["movie_id", column]].head(5)

    outlier_rows.append(
        {
            "field": column,
            "iqr_lower_bound": lower_bound,
            "iqr_upper_bound": upper_bound,
            "values_outside_iqr": int(unusual.sum()),
            "negative_values": int(valid_values.lt(0).sum()),
            "example_movie_values": [
                f"{row.movie_id}: {row_value}"
                for row, row_value in zip(
                    examples.itertuples(index=False),
                    examples[column],
                    strict=True,
                )
            ],
        }
    )

outlier_review = pd.DataFrame(outlier_rows)
display(outlier_review)


,count,mean,std,min,1%,25%,50%,75%,90%,99%,max
budget,112.0,1.084460e+08,8.278818e+07,1.200000e+06,4.665000e+06,3.808750e+07,9.100000e+07,1.500000e+08,2.290000e+08,3.498400e+08,3.790000e+08
popularity,112.0,1.965738e+01,1.344980e+01,4.583000e-01,1.059087e+00,1.035487e+01,1.630640e+01,2.396770e+01,3.680382e+01,6.573367e+01,8.911770e+01
revenue,112.0,5.996588e+08,4.362467e+08,7.390108e+06,2.950291e+07,2.953463e+08,5.195630e+08,8.726984e+08,1.057281e+09,1.993719e+09,2.942650e+09
runtime,112.0,1.304911e+02,2.553676e+01,8.100000e+01,8.188000e+01,1.150000e+02,1.285000e+02,1.432500e+02,1.647000e+02,2.003400e+02,2.020000e+02
vote_average,112.0,7.588705e+00,7.134737e-01,5.752000e+00,6.035910e+00,7.201250e+00,7.720000e+00,8.164500e+00,8.451000e+00,8.675350e+00,8.734000e+00
vote_count,112.0,1.753832e+04,8.270041e+03,4.136000e+03,5.959500e+03,1.103900e+04,1.584100e+04,2.282150e+04,2.926520e+04,4.071385e+04,4.192500e+04


,field,iqr_lower_bound,iqr_upper_bound,values_outside_iqr,negative_values,example_movie_values
0,budget,-1.297812e+08,3.178688e+08,2,0,"[299534: 356000000, 1865: 379000000]"
1,popularity,-1.006436e+01,4.438694e+01,6,0,"[155: 66.8926, 278: 45.4678, 299534: 89.1177, ..."
2,revenue,-5.706817e+08,1.738726e+09,2,0,"[299534: 2942649778, 299536: 2052415039]"
3,runtime,7.262500e+01,1.856250e+02,3,0,"[122: 201, 240: 202, 424: 195]"
4,vote_average,5.756375e+00,9.609375e+00,1,0,[41513: 5.752]
5,vote_count,-6.634750e+03,4.049525e+04,2,0,"[27205: 40754, 157336: 41925]"


## 4. Release dates and common categories

Review coverage and the observed range before defining date rules. Frequency tables help identify dominant categories, spelling/label variants, or unexpected missing values.


In [17]:
if "release_date" in movies.columns:
    release_dates = movies["release_date"].dropna()
    print(f"Movies with a release date: {len(release_dates):,} / {len(movies):,}")
    if not release_dates.empty:
        print(f"Observed release-date range: {release_dates.min()} to {release_dates.max()}")
        release_year_counts = (
            release_dates.dt.year.value_counts().sort_index().rename("movie_count")
        )
        print("Earliest release years:")
        display(release_year_counts.head(15).to_frame())
        print("Latest release years:")
        display(release_year_counts.tail(15).to_frame())
else:
    print("No normalized release_date column is available.")

category_columns = [
    column
    for column in ("status", "original_language", "adult", "video")
    if column in movies.columns
]
category_rows = []
for column in category_columns:
    counts = movies[column].value_counts(dropna=False).head(15)
    for value, count in counts.items():
        category_rows.append(
            {
                "field": column,
                "value": value,
                "movie_count": int(count),
                "percent_of_movies": (
                    round(count / len(movies) * 100, 2) if len(movies) else pd.NA
                ),
            }
        )

category_profile = pd.DataFrame(category_rows)
display(category_profile)


Movies with a release date: 112 / 112
Observed release-date range: 1966-12-23 00:00:00 to 2019-06-19 00:00:00
Earliest release years:


,movie_count
release_date,
1966,1
1972,1
1974,1
1975,1
1977,1
1979,1
1980,1
1981,1
1982,1


Latest release years:


,movie_count
release_date,
2004,6
2005,3
2006,5
2007,6
2008,4
2009,3
2010,5
2011,4
2012,3


,field,value,movie_count,percent_of_movies
0,status,Released,112,100.00
1,original_language,en,110,98.21
2,original_language,ja,1,0.89
3,original_language,it,1,0.89
4,adult,False,112,100.00
5,video,False,112,100.00


## 5. Profile relationships

Compare relationship counts to all movies, including movies with zero links. Then inspect the most common genres. The left join preserves links even if a reference label is missing; the match count makes that case visible.


In [18]:
relationship_tables = {
    "genres": tables.movie_genres,
    "production companies": tables.movie_companies,
    "origin countries": tables.movie_origin_countries,
    "production countries": tables.movie_production_countries,
    "spoken languages": tables.movie_spoken_languages,
    "cast credits": tables.cast_credits,
    "crew credits": tables.crew_credits,
}

relationship_rows = []
for relationship_name, links in relationship_tables.items():
    counts_per_movie = links.groupby("movie_id").size().reindex(movies["movie_id"], fill_value=0)
    relationship_rows.append(
        {
            "relationship": relationship_name,
            "link_rows": len(links),
            "movies_with_at_least_one": int(counts_per_movie.gt(0).sum()),
            "movies_with_zero": int(counts_per_movie.eq(0).sum()),
            "median_links_per_movie_including_zero": (
                float(counts_per_movie.median()) if len(movies) else pd.NA
            ),
            "maximum_links_for_one_movie": (
                int(counts_per_movie.max()) if len(movies) else pd.NA
            ),
        }
    )

relationship_profile = pd.DataFrame(relationship_rows)
display(relationship_profile)

genre_counts = (
    tables.movie_genres.groupby("genre_id")
    .size()
    .rename("movie_count")
    .reset_index()
    .merge(
        tables.genres,
        on="genre_id",
        how="left",
        validate="many_to_one",
        indicator="genre_match",
    )
    .sort_values("movie_count", ascending=False)
)
display(genre_counts.head(20))
print("Genre-reference matches:")
display(genre_counts["genre_match"].value_counts(dropna=False).to_frame("link_rows"))


,relationship,link_rows,movies_with_at_least_one,movies_with_zero,median_links_per_movie_including_zero,maximum_links_for_one_movie
0,genres,320,112,0,3.0,5
1,production companies,307,112,0,3.0,7
2,origin countries,117,112,0,1.0,2
3,production countries,154,112,0,1.0,4
4,spoken languages,219,112,0,1.0,6
5,cast credits,7827,112,0,65.5,177
6,crew credits,24929,112,0,182.0,953


,genre_id,movie_count,genre_name,genre_match
5,28,61,Action,both
0,12,60,Adventure,both
11,878,36,Science Fiction,both
9,53,33,Thriller,both
1,14,27,Fantasy,both
3,18,19,Drama,both
6,35,19,Comedy,both
10,80,18,Crime,both
2,16,17,Animation,both
14,10751,17,Family,both


Genre-reference matches:


,link_rows
genre_match,
both,16
left_only,0
right_only,0


## 6. Turn the profile into follow-up work

Use this checklist to complete the profiling cycle:

- **Confirm grain:** check that every table has the intended one-row-per-entity, one-row-per-link, or one-row-per-credit meaning.
- **Investigate missingness:** determine whether each missing field is optional, not collected, or a pipeline defect. Do not fill missing values without a reason.
- **Review unusual measures:** inspect IQR examples and negative counts against source records and domain context; do not treat statistical outliers as automatic errors.
- **Review date/category coverage:** verify unexpected periods, labels, and missing categories with a data owner before setting accepted ranges.
- **Check relationships:** resolve unmatched reference IDs and understand movies with no links where coverage is expected.
- **Write explicit rules:** once business expectations are agreed, add named checks and thresholds to the project's staged data checks rather than silently cleaning this notebook's working copy.

Re-run from the top whenever the cache changes. The tables and summaries are generated from the local cache each time; no intermediate output is written back to the data files.
